# ============================================================
# STEP 2B — ADVERSARIAL FEATURES (38 features)
# Turkish Quishing Detection v10
# ============================================================
# Adds 38 adversarial / evasion-detection features on top of
# the 58 lexical features → 96 total.
#
# Designed to catch what structural features miss:
#   - Brand impersonation (garanti-login.xyz)
#   - Typo-squatting (garranti, akhbank)
#   - Homograph / mixed-script (раypal with Cyrillic)
#   - Subdomain abuse (garanti.com.tr.evil.xyz)
#   - Obfuscation in other_malicious (hash paths, encoding)
#   - Suspicious TLD + brand combos
#
# Input  : lexical_features_v10.csv  (58 features)
# Output : features_full_v10.csv     (96 features)
#          adversarial_feature_names.pkl
# ============================================================

In [ ]:
import os, re, math, pickle, warnings
from collections import Counter
from urllib.parse import urlparse, unquote
from turkish_lexicons import (TR_PHISHING, TR_SCAM, TR_MALWARE,
                              TR_DEFACEMENT, TR_URGENCY, TR_BRANDS)
import numpy as np
import pandas as pd
import tldextract
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")

# ── PATHS ─────────────────────────────────────────────────────
DATA_DIR = ("/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May")
INPUT_FILE  = os.path.join(DATA_DIR, "lexical_features_final.csv")
OUTPUT_CSV  = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_PKL  = os.path.join(DATA_DIR, "adversarial_feature_names_final.pkl")
#═══════════════════════════════════════════════════════

GLOBAL_BRANDS =[
        # Big Tech
    "microsoft",
    "google",
    "apple",
    "amazon",
    "meta",
    "facebook",
    "instagram",
    "whatsapp",
    "x",
    "twitter",
    "tiktok",
    "snapchat",
    "linkedin",
    "reddit",
    "discord",
    "telegram",
    "wechat",

    # Productivity / Cloud
    "office365",
    "outlook",
    "onedrive",
    "icloud",
    "dropbox",
    "adobe",
    "canva",
    "notion",
    "slack",
    "zoom",

    # Streaming / Entertainment
    "netflix",
    "spotify",
    "youtube",
    "disneyplus",
    "primevideo",
    "twitch",

    # Payments / Fintech
    "paypal",
    "wise",
    "payoneer",
    "stripe",
    "skrill",
    "revolut",
    "westernunion",

    # Crypto
    "binance",
    "coinbase",
    "metamask",
    "kraken",
    "okx",
    "bybit",
    "trustwallet",
    "blockchain",

    # Logistics / Shipping
    "dhl",
    "fedex",
    "ups",
    "aramex",
    "usps",
    "royalmail",

    # Ecommerce / Marketplaces
    "ebay",
    "aliexpress",
    "etsy",
    "shopify",

    # Travel / Booking
    "booking",
    "airbnb",
    "uber",
    "lyft",

    # Telecom / Devices
    "samsung",
    "huawei",
    "xiaomi",
    "sony",

    # Gaming
    "steam",
    "epicgames",
    "playstation",
    "xbox",
    "nintendo",

    # Security / Authentication
    "norton",
    "mcafee",
    "lastpass",
    "1password",
    "okta",

    # AI / Developer
    "openai",
    "github",
    "gitlab",
    # Search / AI / Browsers
    "chatgpt",
    "gemini",
    "copilot",
    "claude",
    "perplexity",
    "bing",
    "yahoo",
    "duckduckgo",
    "firefox",
    "chrome",
    "safari",
    "opera",
    "brave",

    # Email / Communication
    "gmail",
    "protonmail",
    "yandexmail",
    "hotmail",
    "messenger",
    "signal",
    "viber",
    "line",
    "skype",

    # Cloud / Infrastructure
    "aws",
    "azure",
    "gcp",
    "digitalocean",
    "cloudflare",
    "vercel",
    "heroku",

    # Developer / Engineering
    "stackoverflow",
    "huggingface",
    "docker",
    "kubernetes",
    "jira",
    "atlassian",
    "bitbucket",
    "postman",

    # Finance / Trading
    "visa",
    "mastercard",
    "americanexpress",
    "amex",
    "paytm",
    "cashapp",
    "venmo",
    "zelle",
    "robinhood",
    "etrade",
    "interactivebrokers",

    # Crypto / Web3
    "phantom",
    "ledger",
    "trezor",
    "kucoin",
    "crypto",
    "cryptocom",
    "gateio",
    "mexc",
    "bitfinex",
    "pancakeswap",
    "uniswap",

    # Retail / Consumer
    "walmart",
    "target",
    "costco",
    "ikea",
    "zara",
    "hm",
    "uniqlo",
    "nike",
    "adidas",
    "puma",

    # Food / Delivery
    "ubereats",
    "doordash",
    "grubhub",
    "deliveroo",
    "glovo",
    "instacart",
    "starbucks",
    "mcdonalds",
    "kfc",
    "burgerking",

    # Travel / Mobility
    "expedia",
    "tripadvisor",
    "kayak",
    "skyscanner",
    "bolt",
    "bookingcom",

    # Media / Social Content
    "pinterest",
    "patreon",
    "onlyfans",
    "medium",
    "substack",
    "quora",

    # Gaming / Esports
    "riotgames",
    "valorant",
    "leagueoflegends",
    "minecraft",
    "roblox",
    "fortnite",
    "ea",
    "ubisoft",

    # Telecom / Hardware
    "intel",
    "amd",
    "nvidia",
    "qualcomm",
    "tesla",

    # Security / Password Managers
    "dashlane",
    "bitwarden",
    "authy",
    "duo",

    # Enterprise / SaaS
    "salesforce",
    "hubspot",
    "zendesk",
    "shopifyplus",
    "mailchimp",

    # Education / Learning
    "coursera",
    "udemy",
    "duolingo",
    "khanacademy",

    # File Sharing / Storage
    "wetransfer",
    "mega",
    "box",
     # Additional Big Tech / Internet
    "baidu",
    "alibaba",
    "tencent",
    "qq",
    "naver",
    "rakuten",
    "vk",
    "mailru",

    # Collaboration / Meetings
    "teams",
    "webex",
    "gotomeeting",
    "miro",
    "figma",
    "airtable",
    "clickup",
    "asana",
    "trello",

    # Hosting / Domains
    "godaddy",
    "namecheap",
    "hostinger",
    "bluehost",
    "cpanel",

    # CDN / Networking
    "akamai",
    "fastly",
    "nginx",

    # Databases / Data
    "mongodb",
    "redis",
    "mysql",
    "postgresql",
    "snowflake",
    "databricks",

    # DevOps / Monitoring
    "jenkins",
    "terraform",
    "ansible",
    "grafana",
    "prometheus",
    "datadog",
    "newrelic",
    "sentry",

    # Cybersecurity
    "crowdstrike",
    "fortinet",
    "paloaltonetworks",
    "checkpoint",
    "kaspersky",
    "avast",
    "avg",
    "eset",
    "sophos",

    # Authentication / Identity
    "auth0",
    "pingidentity",
    "onelogin",
    "yubikey",

    # Additional Finance / Payments
    "worldpay",
    "adyen",
    "klarna",
    "afterpay",
    "affirm",
    "discover",
    "capitalone",
    "hsbc",
    "barclays",
    "chase",
    "wellsfargo",

    # Additional Crypto / Exchanges
    "bitstamp",
    "gemini",
    "binanceus",
    "trustwallet",
    "exodus",
    "electrum",
    "metatrader",

    # Retail / Marketplaces
    "flipkart",
    "mercadolibre",
    "taobao",
    "tmall",
    "wish",
    "newegg",
    "bestbuy",

    # Transportation / Automotive
    "bmw",
    "mercedes",
    "audi",
    "toyota",
    "ford",
    "chevrolet",
    "volkswagen",
    "hyundai",

    # Airlines / Travel
    "emirates",
    "qatarairways",
    "lufthansa",
    "turkishairlines",
    "ryanair",
    "easyjet",

    # Hospitality
    "hilton",
    "marriott",
    "hyatt",
    "accor",

    # Media / Streaming / Music
    "soundcloud",
    "deezer",
    "tidal",
    "hulu",
    "hbo",
    "max",
    "paramountplus",
    "peacock",

    # News / Publishing
    "bbc",
    "cnn",
    "nytimes",
    "forbes",
    "bloomberg",

    # Gaming Platforms
    "battle",
    "battlenet",
    "origin",
    "gog",
    "rockstargames",

    # Smart Devices / IoT
    "philips",
    "dyson",
    "ring",
    "arlo",

    # Fashion / Luxury
    "gucci",
    "louisvuitton",
    "prada",
    "chanel",
    "rolex",

    # Food / Beverage
    "cocacola",
    "pepsi",
    "nestle",
    "redbull",
    "dominos",
    "subway",

    # Healthcare / Pharma
    "pfizer",
    "moderna",
    "johnsonandjohnson",

    # Productivity / Notes
    "evernote",
    "obsidian",
    "todoist",

    # AI / Automation
    "midjourney",
    "stabilityai",
    "runway",
    "zapier",
    "ifttt",
]
ALL_BRANDS = (TR_BRANDS +
              GLOBAL_BRANDS)

SUSPICIOUS_TLDS = { "xyz","top","club","online","site","website","space","info","click","link","live","stream","work","pw","tk","ml","ga","cf","lat","cfd","sbs","icu","rest","fit","monster","quest"}

LEGIT_TR_TLDS = { "tr", "com.tr", "net.tr","org.tr","edu.tr","gov.tr","bel.tr","k12.tr","av.tr","bbs.tr","biz.tr","dr.tr","gen.tr","info.tr","name.tr","pol.tr","tel.tr","tv.tr","web.tr","kep.tr",}


# ════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════
def levenshtein(a, b):
    """Edit distance between two strings (for typo detection)."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        cur = [i + 1]
        for j, cb in enumerate(b):
            cost = 0 if ca == cb else 1
            cur.append(min(prev[j+1] + 1, cur[j] + 1, prev[j] + cost))
        prev = cur
    return prev[-1]

def min_brand_distance(token, brands, max_check=2):
    """Smallest edit distance from token to any brand (typo-squat signal)."""
    if not token or len(token) < 3:
        return 99
    best = 99
    for b in brands:
        if abs(len(token) - len(b)) > max_check + 1:
            continue
        d = levenshtein(token, b)
        if d < best:
            best = d
            if best == 0:
                return 0
    return best
def has_legit_tr_tld(hostname):
    hostname = hostname.lower().rstrip(".")
    return any(hostname == tld or hostname.endswith("." + tld) for tld in LEGIT_TR_TLDS)
def shannon(s):
    s = str(s)
    if not s:
        return 0.0
    freq = Counter(s)
    n = len(s)
    return -sum((c/n) * math.log2(c/n) for c in freq.values())

def is_mostly_consonants(s):
    """Random/DGA domains have unusual consonant runs."""
    if not s:
        return 0
    vowels = sum(c in "aeiou" for c in s.lower())
    return int(vowels / max(len(s), 1) < 0.25)

# ════════════════════════════════════════════════════════════
# 38 ADVERSARIAL FEATURE NAMES
# ════════════════════════════════════════════════════════════
ADVERSARIAL_FEATURE_NAMES = [
    # Brand impersonation (8)
    "contains_brand","brand_in_subdomain","brand_in_path",
    "brand_not_in_domain","brand_tld_mismatch",
    "num_brands_mentioned","brand_with_hyphen","brand_plus_keyword",
    # Typo-squatting (5)
    "min_brand_edit_dist","is_typo_squat","has_char_substitution",
    "has_doubled_char","brand_homoglyph",
    # Subdomain abuse (5)
    "excessive_subdomains","tr_in_subdomain","com_in_subdomain",
    "brand_dot_in_subdomain","deep_subdomain_nesting",
    # Suspicious TLD combos (4)
    "susp_tld_with_brand","susp_tld_with_keyword",
    "short_domain_susp_tld","numeric_with_susp_tld",
    # Obfuscation / encoding (7)
    "pct_encoded_ratio","has_unicode_escape","has_punycode",
    "hex_in_domain","excessive_hyphens","random_looking_domain",
    "consonant_heavy_domain",
    # Path / hash signals (5)
    "long_random_path","hash_like_segment","base64_like_segment",

    "many_path_dirs","suspicious_file_in_path",
    # Redirect / nesting (4)
    "has_url_in_url","has_redirect_param","at_symbol_trick",
    "double_protocol",
]
assert len(ADVERSARIAL_FEATURE_NAMES) == 38, len(ADVERSARIAL_FEATURE_NAMES)

# ════════════════════════════════════════════════════════════
# EXTRACT 38 ADVERSARIAL FEATURES
# ════════════════════════════════════════════════════════════
def extract_adversarial(url):
    u       = str(url).strip()
    u_low   = u.lower()
    ext     = tldextract.extract(u if u.startswith("http") else "http://" + u)
    try:
        parsed = urlparse(u if u.startswith("http") else "http://" + u)
    except Exception:
        parsed = None

    subdomain = (ext.subdomain or "").lower()
    domain    = (ext.domain or "").lower()
    suffix    = (ext.suffix or "").lower()
    path      = (parsed.path if parsed else "").lower()
    netloc    = (parsed.netloc if parsed else "").lower()
    full_low  = u_low

    f = {}

    # ── Brand impersonation (8) ──────────────────────────────
    brands_found = [b for b in ALL_BRANDS if b in full_low]
    f["contains_brand"]      = int(len(brands_found) > 0)
    f["brand_in_subdomain"]  = int(any(b in subdomain for b in ALL_BRANDS))
    f["brand_in_path"]       = int(any(b in path for b in ALL_BRANDS))
    # Brand appears but NOT as the registered domain = impersonation
    f["brand_not_in_domain"] = int(
        len(brands_found) > 0 and
        not any(b == domain for b in ALL_BRANDS))
    # Brand present but TLD is not a legit TR TLD
    full_tld = f"{domain}.{suffix}"
    f["brand_tld_mismatch"]  = int(
        len(brands_found) > 0 and suffix in SUSPICIOUS_TLDS)
    f["num_brands_mentioned"] = len(brands_found)
    f["brand_with_hyphen"]    = int(any(
        f"{b}-" in full_low or f"-{b}" in full_low for b in ALL_BRANDS))
    PHISH_WORDS = ["giris","giriş","odeme","ödeme","hesap","banka","kripto",
    "dogrula","doğrula","guvenli","güvenli","sifre","şifre",
    "uyelik","üyelik","kargo","fatura","kampanya","indirim",
    "kazandin","kazandın","hediye","odul","ödül","para"]
    f["brand_plus_keyword"]   = int(
        len(brands_found) > 0 and
        any(w in full_low for w in PHISH_WORDS))

    # ── Typo-squatting (5) ───────────────────────────────────
    dist = min_brand_distance(domain, ALL_BRANDS)
    f["min_brand_edit_dist"]   = min(dist, 10)
    f["is_typo_squat"]         = int(1 <= dist <= 2)  # close but not exact
    # Character substitution (0→o, 1→l, 3→e, etc.)
    f["has_char_substitution"] = int(bool(
        re.search(r"[a-z]+[013457]+[a-z]+", domain)))
    f["has_doubled_char"]      = int(bool(re.search(r"(.)\1", domain))
                                     and dist <= 3)
    # Mixed-script homoglyph (latin + cyrillic/greek)
    f["brand_homoglyph"] = int(any(
        ord(c) > 127 for c in domain))

    # ── Subdomain abuse (5) ──────────────────────────────────
    sub_parts = subdomain.split(".") if subdomain else []
    f["excessive_subdomains"]   = int(len(sub_parts) >= 3)
    f["tr_in_subdomain"]        = int(
        "tr" in sub_parts or "com" in sub_parts)
    f["com_in_subdomain"]       = int("com" in sub_parts)
    # garanti.com.tr.evil.xyz — legit domain embedded in subdomain
    f["brand_dot_in_subdomain"] = int(any(
        b in subdomain for b in ALL_BRANDS) and len(sub_parts) >= 2)
    f["deep_subdomain_nesting"] = min(len(sub_parts), 6)

    # ── Suspicious TLD combos (4) ────────────────────────────
    is_susp_tld = suffix in SUSPICIOUS_TLDS
    f["susp_tld_with_brand"]   = int(is_susp_tld and len(brands_found) > 0)
    f["susp_tld_with_keyword"] = int(
        is_susp_tld and any(w in full_low for w in PHISH_WORDS))
    f["short_domain_susp_tld"] = int(is_susp_tld and len(domain) <= 6)
    f["numeric_with_susp_tld"] = int(
        is_susp_tld and any(c.isdigit() for c in domain))

    # ── Obfuscation / encoding (7) ───────────────────────────
    n_url = max(len(u), 1)
    f["pct_encoded_ratio"]   = round(
        len(re.findall(r"%[0-9a-fA-F]{2}", u)) / n_url, 4)
    f["has_unicode_escape"]  = int("\\u" in u or "%u" in u_low)
    f["has_punycode"]        = int("xn--" in full_low)
    f["hex_in_domain"]       = int(bool(
        re.search(r"[0-9a-f]{8,}", domain)))
    f["excessive_hyphens"]   = int(domain.count("-") >= 3)
    f["random_looking_domain"] = int(
        shannon(domain) > 3.5 and len(domain) >= 10)
    f["consonant_heavy_domain"] = is_mostly_consonants(domain)

    # ── Path / hash signals (7) — targets other_malicious ───
    path_segs = [p for p in path.split("/") if p]
    f["long_random_path"] = int(
        any(len(seg) >= 20 and shannon(seg) > 3.5 for seg in path_segs))
    f["hash_like_segment"] = int(any(
        bool(re.fullmatch(r"[0-9a-f]{16,}", seg)) for seg in path_segs))
    f["base64_like_segment"] = int(any(
        len(seg) >= 12 and bool(re.fullmatch(r"[A-Za-z0-9+/=_-]+", seg))
        and shannon(seg) > 4.0 for seg in path_segs))
    f["many_path_dirs"] = int(len(path_segs) >= 5)
    f["suspicious_file_in_path"] = int(bool(re.search(
        r"\.(exe|zip|rar|apk|msi|bat|dll|scr|jar|php|js)(\?|$|/)",
        path)))

    # ── Redirect / nesting (4) ───────────────────────────────
    f["has_url_in_url"] = int(
        full_low.count("http") > 1 or "%2f%2f" in full_low)
    f["has_redirect_param"] = int(bool(re.search(
        r"[?&](redirect|url|next|return|goto|target|dest|continue)=",
        full_low)))
    f["at_symbol_trick"]  = int("@" in netloc)
    f["double_protocol"]  = int(bool(re.search(r"https?:.*https?:", full_low)))

    return {k: f[k] for k in ADVERSARIAL_FEATURE_NAMES}


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 2B — ADVERSARIAL FEATURES (38 features → 96 total)")
print("=" * 70)

print(f"\n[1] Loading lexical features ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"    Rows: {len(df):,}  Cols: {len(df.columns)}")

if "url" not in df.columns:
    raise ValueError("Need 'url' column from Step 2 output")

# Sanity check
sample = df["url"].iloc[0]
sf = extract_adversarial(sample)
print(f"\n[2] Sanity check on: '{sample[:55]}'")
print(f"    Adversarial features: {len(sf)} ✓")

# Test on known phishing patterns
print(f"\n[3] Validation on synthetic phishing URLs:")
tests = [
    "http://garanti-giris.xyz/hesap-dogrula",
    "https://garanti.com.tr.secure-login.tk/",
    "http://paypa1.com/login",            # typo: paypal→paypa1
    "https://akhbank.xyz/giris",          # typo: akbank→akhbank
    "http://osbases.tribun-triptych.lat/k5s8-byna-tqed-r6mwn",  # C2
]
for t in tests:
    af = extract_adversarial(t)
    flags = [k for k,v in af.items() if v and v != 99]
    print(f"    {t[:48]:<48s}")
    print(f"      → {', '.join(flags[:6])}")

# Extract all
print(f"\n[4] Extracting adversarial features for {len(df):,} URLs ...")
print(f"    (Levenshtein is slow — expect 10-20 min on 1.24M)")

records = []
for i, url in enumerate(df["url"]):
    records.append(extract_adversarial(url))
    if (i + 1) % 50_000 == 0:
        print(f"    {i+1:>9,} / {len(df):,}  ({(i+1)/len(df)*100:5.1f}%)")

adv_df = pd.DataFrame(records)
print(f"\n[5] Adversarial matrix: {adv_df.shape}  Nulls: {adv_df.isnull().sum().sum()}")

# Combine with lexical
print(f"\n[6] Merging lexical + adversarial ...")
LEXICAL_COLS = [c for c in df.columns if c not in
                ["url","source","tld","label","label_enc",
                 "class_final","class_enc","fold","reg_domain"]]
full_df = pd.concat([
    df[LEXICAL_COLS].reset_index(drop=True),
    adv_df.reset_index(drop=True),
], axis=1)
# Re-attach metadata
for c in ["url","source","tld","label","label_enc",
          "class_final","class_enc","fold","reg_domain"]:
    if c in df.columns:
        full_df[c] = df[c].values

print(f"    Total features: {len(LEXICAL_COLS) + len(ADVERSARIAL_FEATURE_NAMES)}")
print(f"    ({len(LEXICAL_COLS)} lexical + {len(ADVERSARIAL_FEATURE_NAMES)} adversarial)")

# Class verification
print(f"\n[7] Adversarial feature rates by class:")
adv_df["class_final"] = df["class_final"].values
key_adv = ["contains_brand","is_typo_squat","susp_tld_with_brand",
           "hash_like_segment","long_random_path","brand_tld_mismatch",
           "has_redirect_param","random_looking_domain"]
print(adv_df.groupby("class_final")[key_adv].mean().round(3).to_string())

# Save
print(f"\n[8] Saving ...")
full_df.to_csv(OUTPUT_CSV, index=False)
print(f"    {OUTPUT_CSV}")
with open(OUTPUT_PKL, "wb") as fh:
    pickle.dump(ADVERSARIAL_FEATURE_NAMES, fh)
print(f"    {OUTPUT_PKL}")

# Mutual information of adversarial features
print(f"\n[9] MI ranking — adversarial only (binary) ...")
X_adv = adv_df[ADVERSARIAL_FEATURE_NAMES].values
y_bin = df["label_enc"].values
mi = mutual_info_classif(X_adv, y_bin, random_state=42)
mi_s = pd.Series(mi, index=ADVERSARIAL_FEATURE_NAMES).sort_values(ascending=False)
print(mi_s.head(12).round(4).to_string())

print(f"\n[10] MI ranking — adversarial only (5-class) ...")
y_5c = df["class_enc"].values
mi5 = mutual_info_classif(X_adv, y_5c, random_state=42)
mi5_s = pd.Series(mi5, index=ADVERSARIAL_FEATURE_NAMES).sort_values(ascending=False)
print(mi5_s.head(12).round(4).to_string())

print(f"""
{'='*70}
STEP 2B COMPLETE — 96 total features
{'='*70}
  Lexical     : 58
  Adversarial : 38
  Total       : 96

  Top adversarial (binary): {mi_s.index[0]} (MI={mi_s.iloc[0]:.4f})
  Top adversarial (5class): {mi5_s.index[0]} (MI={mi5_s.iloc[0]:.4f})

  Output: {OUTPUT_CSV}

  Next → Step 3: 6-model CV with 3 experiment variants
    Exp A: all 96 features
    Exp B: drop is_tr_domain
    Exp C: drop is_tr_domain + is_https (honest deployable estimate)
{'='*70}
""")

STEP 2B — ADVERSARIAL FEATURES (38 features → 96 total)

[1] Loading lexical features ...
    Rows: 1,239,308  Cols: 67

[2] Sanity check on: 'http://afsaces.accesscam.org'
    Adversarial features: 38 ✓

[3] Validation on synthetic phishing URLs:
    http://garanti-giris.xyz/hesap-dogrula          
      → contains_brand, brand_not_in_domain, brand_tld_mismatch, num_brands_mentioned, brand_with_hyphen, brand_plus_keyword
    https://garanti.com.tr.secure-login.tk/         
      → contains_brand, brand_in_subdomain, brand_not_in_domain, brand_tld_mismatch, num_brands_mentioned, min_brand_edit_dist
    http://paypa1.com/login                         
      → min_brand_edit_dist, is_typo_squat
    https://akhbank.xyz/giris                       
      → contains_brand, brand_not_in_domain, brand_tld_mismatch, num_brands_mentioned, brand_plus_keyword, min_brand_edit_dist
    http://osbases.tribun-triptych.lat/k5s8-byna-tqe
      → min_brand_edit_dist, deep_subdomain_nesting, consonant_he